# Configure a command-line workflow

Start with an AnnData file containing raw counts, a time label, cell-type labels,
and spatial coordinates. This guide shows how to tell CytoBridge where those
fields are stored, then run preprocessing, training, and analysis with one
command. To work through the Python functions individually, use
[Train a model](../training.md).

The [small preprocessing example](data_preparation/synthetic_preprocessing.ipynb)
shows how to create an AnnData object. To reuse an existing model, see
[Continue from a trained model](../reuse_model.md).

## Describe your input

In the example below, counts are in `layers['counts']`, time is in
`obs['stage']`, cell type is in `obs['cell_type']`, and two-dimensional
coordinates are in `obsm['spatial']`.

Use raw, non-negative counts, unique observation names, and gene symbols
matching your LR table. Do not normalize or run PCA first: preprocessing does
that.

## Create a configuration

The following cells load an example configuration, change its input fields for
observations at stages D0, D2, and D4, and show the settings before you save it.
Edit the values in these cells for your experiment. They create a new
configuration rather than editing an existing JSON file.

In [1]:
from pathlib import Path
import json
import os
import pandas as pd
import yaml
from importlib.resources import files

from CytoBridge.workflow import load_workflow_config

PROJECT_DIR = Path(os.environ.get("CYTOBRIDGE_PROJECT_DIR", ".")).resolve()
CONFIG_PATH = PROJECT_DIR / "configs/my_dataset.json"
TRAINING_CONFIG_PATH = PROJECT_DIR / "configs/my_dataset.yaml"
config, _ = load_workflow_config("zebrafish")
training_config = yaml.safe_load(
    files("CytoBridge").joinpath("configs", config["train"]["config"]).read_text())

# Start a new experiment without the paper benchmark identifiers.
training_config.pop("matched_ablation", None)
config["dataset"]["name"] = "my_dataset"
config["preprocess"]["time_key"] = "stage"
config["preprocess"]["annotation_source"] = "cell_type"
config["preprocess"]["batch_indices"] = None  # Use all observed stages.
align = config["preprocess"]["align"]
align["expression_layer"] = "counts"
align["input_spatial_key"] = "spatial"
align.pop("spatial_obs_keys", None)
align["time_mapping"] = {"D0": 0, "D2": 1, "D4": 2}
config["downstream"]["observed"] = [0, 1, 2]
config["downstream"]["interpolated"] = [0.5, 1.5]

These are example times, not values to copy unchanged. Replace them with your
measured stages and the times you want to predict. Keep the ordering and
relative spacing of the observed times consistent with the experiment.
The example removes the Zebrafish-specific stage selection so that all three
stages are used.

The table below lists the main settings to review before saving. In particular,
choose a suitable LR database and species, spatial scale, and neighborhood
size. The dataset configurations contain the paper's settings. The
[dataset tutorials](dataset_workflows/index.md) cover analysis after training.

In [2]:
pd.DataFrame({
    "Setting": [
        "Count layer", "Time column", "Cell-type column", "Coordinates",
        "Time mapping", "LR species", "Interaction neighborhood",
    ],
    "Value": [
        align["expression_layer"],
        config["preprocess"]["time_key"],
        config["preprocess"]["annotation_source"],
        align["input_spatial_key"],
        str(align["time_mapping"]),
        config["downstream"].get("preferred_species_tag"),
        config["train"].get("interaction_cutoff"),
    ],
})

,Setting,Value
0,Count layer,counts
1,Time column,stage
2,Cell-type column,cell_type
3,Coordinates,spatial
4,Time mapping,"{'D0': 0, 'D2': 1, 'D4': 2}"
5,LR species,zebrafish
6,Interaction neighborhood,0.096064


After editing the values, save both configuration files. The JSON describes the
input and workflow. The YAML contains the network and training settings. This
cell keeps the interaction neighborhood and edge threshold consistent between
them and makes the workflow use your new YAML.

In [3]:
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
interaction = training_config["model"]["interaction_net"]
interaction["cutoff"] = config["train"]["interaction_cutoff"]
interaction["edge_predictor_thre"] = config["train"]["edge_predictor_threshold"]
TRAINING_CONFIG_PATH.write_text(yaml.safe_dump(training_config, sort_keys=False))
config["train"]["config"] = str(TRAINING_CONFIG_PATH)
CONFIG_PATH.write_text(json.dumps(config, indent=2) + "\n")
CONFIG_PATH

PosixPath('/private/var/folders/b5/t2zljhrs6vq96hjn5_n9zzz80000gn/T/cytobridge-config-97k6ybgn/configs/my_dataset.json')

The configuration above starts from Zebrafish settings. For another species,
supply your LR CSV in the command below and set
`downstream.preferred_species_tag` to its species tag. For graph construction,
the CSV needs `Ligand`, `Receptor`, `Pathway`, and `Annotation` columns, as in
the included CellChatDB tables. Keep the interaction-type annotations from
the database.

## Train and calculate results

Put your input at `data/my_dataset_raw.h5ad` inside your project folder. In a
terminal, change to that folder (the `PROJECT_DIR` shown above), then run:

```bash
cytobridge workflow --config configs/my_dataset.json --train   --input-h5ad data/my_dataset_raw.h5ad   --output-dir outputs/my_dataset --device cuda
```

This **one command** preprocesses the input, fits the LR edge predictor, trains
the model, and calculates the downstream results. Do not run `preprocess`
first.

If you are using a custom LR table, use this version of the same command:

```bash
cytobridge workflow --config configs/my_dataset.json --train   --input-h5ad data/my_dataset_raw.h5ad   --graph-database data/my_ligand_receptor_table.csv   --lr-database data/my_ligand_receptor_table.csv   --output-dir outputs/my_dataset --device cuda
```

Choose one of these commands. The second supplies the same LR table for
training and downstream analysis.

## Open the results

The trained model is in `outputs/my_dataset/training/`. Growth, velocities,
trajectories, and interaction tables are in `outputs/my_dataset/downstream/`.
Open its `summary.json` for the list of analyses and `figures/` for the plots.

To change the downstream analysis without training again, follow
[Continue from a trained model](../reuse_model.md) with your own configuration,
aligned H5AD, and training directory.